In [1]:
import gseapy as gp
from lincs_gsnn.explain.infer_edge_weights import infer_edge_weights
import torch 
import pandas as pd 
from lincs_gsnn.explain.viz import annotate_edges, get_drug_edges, make_subgraph
from pypath.utils import mapping

from matplotlib import pyplot as plt 
import numpy as np 

from lincs_gsnn.models.ODEFunc import ODEFunc

from torchdiffeq import odeint

from scipy.stats import hypergeom
from lincs_gsnn.data.TrajDataset import TrajDataset
from lincs_gsnn.data.DXDTDataset import DXDTDataset
from torch.utils.data import DataLoader

from gsnn.interpret.IGExplainer import IGExplainer
from gsnn.interpret.GSNNExplainer import GSNNExplainer

import seaborn as sbn 

from gsnn.optim.OutputEdgeInferer import OutputEdgeInferer

%load_ext autoreload
%autoreload 2

In [2]:
root1 = '../../workflow_outputs/lincs-gsnn'
root2 = '../../workflow_outputs/lincs-traj'

data = torch.load(f'{root1}/default/bionetwork/bionetwork.pt', weights_only=False)
model = torch.load(f'{root1}/default/pretrain/pretrained_model.pt', weights_only=False).eval()
dxdt_scale = torch.load(f'{root1}/default/pretrain/dxdt_scale.pt', weights_only=False).item()
x_names = pd.read_csv(f'{root2}/runs/exp/default_v02/output/predict_grid/gene_names.csv')['gene_names'].values.astype(str)
dxdt_meta = pd.read_csv(f'{root2}/runs/exp/default_v02/output/predict_grid/dxdt_meta.csv')
x_meta = pd.read_csv(f'{root2}/runs/exp/default_v02/output/predict_grid/pred_meta.csv')

valid_drugs = [x.split('DRUG__')[1] for x in data.node_names_dict['input'] if 'DRUG__' in x]

dxdt_meta = dxdt_meta[dxdt_meta['pert_id'].isin(valid_drugs)] 
x_meta = x_meta[x_meta['pert_id'].isin(valid_drugs)] 

In [3]:
cell_line = 'HME1'

dxdt_meta = dxdt_meta[dxdt_meta['cell_iname'] == cell_line]

In [4]:
dxdt_dir = f'{root2}/runs/exp/default_v02/output/predict_grid/dxdt'

In [5]:
batch_size = 128

train_ids = dxdt_meta.sample(frac=0.8).index 
test_ids = dxdt_meta.index.difference(train_ids) 
train_cond = dxdt_meta.loc[train_ids]
test_cond = dxdt_meta.loc[test_ids]

train_dataset = DXDTDataset(train_cond, 
                        input_names=data.node_names_dict['input'], 
                        output_names=data.node_names_dict['output'], 
                        src_names=x_names, 
                        obs_dir=dxdt_dir, 
                        scale=dxdt_scale) 

test_dataset = DXDTDataset(test_cond, 
                        input_names=data.node_names_dict['input'], 
                        output_names=data.node_names_dict['output'], 
                        src_names=x_names, 
                        obs_dir=dxdt_dir, 
                        scale=dxdt_scale) 

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
OEI = OutputEdgeInferer(data, model.channels*model.layers, lr=1e-2, wd=1e-4, epochs=3, agg='all', use_batchnorm=True)
res = OEI.fit(train_loader, model, device='cuda')

Fitting OutputEdgeInferer on cuda...
# parameters:  78631200


In [ ]:
torch.cuda.empty_cache()

In [ ]:
res  = OEI.evaluate(dataloader=test_loader, model=model, device='cuda', verbose=True)

Evaluating OutputEdgeInferer on cuda...


In [ ]:
res.sort_values('snr', ascending=False).head(10)

,func_node,output_node,mse,r2,r,has_edge,p_value,snr,l1_norm,l2_norm,...,model_r2,model_r,model_mse,r2_gain,r_gain,mse_gain,q_value,snr_rank,sparsity_rank,within_output_rank
1,RNA__SQOR,GENE__ATF6,0.375740,0.816877,0.922088,False,0.000000e+00,3.084258,1.155960,0.419935,...,0.010931,0.219219,2.029414,0.805945,0.702869,-1.653674,0.000000e+00,1,145,57
0,RNA__GNAI2,GENE__PLOD3,0.223829,0.807153,0.913746,False,0.000000e+00,2.963967,0.999034,0.334504,...,0.000000,-0.013159,1.169114,0.807153,0.926905,-0.945285,0.000000e+00,1,169,72
2,RNA__SQOR,GENE__PLOD3,0.227726,0.803795,0.911545,False,0.000000e+00,2.900687,0.902002,0.321583,...,0.000000,-0.013159,1.169114,0.803795,0.924705,-0.941388,0.000000e+00,2,170,73
14,RNA__GNAI2,GENE__ATF6,0.411561,0.799419,0.911337,False,0.000000e+00,2.789248,1.264805,0.418629,...,0.010931,0.219219,2.029414,0.788487,0.692117,-1.617853,0.000000e+00,2,146,58
19,RNA__SQOR,GENE__CNDP2,0.211837,0.783837,0.895236,False,0.000000e+00,2.744496,1.065383,0.310611,...,0.000000,0.016847,0.990386,0.783837,0.878388,-0.778549,0.000000e+00,1,169,56
6,PROTEIN__STAT1,GENE__PACSIN3,0.344224,0.794112,0.911966,False,0.000000e+00,2.711921,1.336898,0.406565,...,0.000093,0.029181,1.671739,0.794018,0.882786,-1.327515,0.000000e+00,1,151,49
12,RNA__MAT2A,GENE__CNDP2,0.206042,0.789750,0.904755,False,0.000000e+00,2.625271,1.000912,0.338132,...,0.000000,0.016847,0.990386,0.789750,0.887907,-0.784344,0.000000e+00,2,170,57
22149,RNA__EPB41L2,GENE__BNIP3L,0.319048,0.789993,0.903793,False,3.427536e-12,2.609538,1.199998,0.404084,...,0.276750,0.725787,1.098783,0.513243,0.178005,-0.779735,1.427843e-11,1,171,93
16,RNA__SQOR,GENE__LRPAP1,0.857902,0.787664,0.907602,False,0.000000e+00,2.605167,1.871002,0.602206,...,0.002758,0.063080,4.029157,0.784905,0.844522,-3.171256,0.000000e+00,1,144,52
4,RNA__SQOR,GENE__ME2,0.703899,0.795647,0.911474,False,0.000000e+00,2.587131,1.763156,0.545902,...,0.000000,-0.004249,3.456414,0.795647,0.915723,-2.752515,0.000000e+00,1,155,38
